In [1]:
import pickle
import re
from itertools import pairwise

import numpy as np

from src.core import Tensor
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
def extract_turns(max_turn_chars=100):
    with open(DATA_FILE, encoding="utf-8") as f:
        text = f.read()

    blocks = re.split(r"\n\s*\n", text.strip())
    turns = []

    for block in blocks:
        lines = block.strip("\n").split("\n")
        if not lines:
            continue

        m = re.compile(r"^([A-Z][A-Za-z' ]{0,30}):\s*$").match(lines[0].strip())
        if not m:
            continue

        speaker = m.group(1).strip()
        content = " ".join(line.strip() for line in lines[1:] if line.strip())
        if not content:
            continue

        turns.append((speaker, content[:max_turn_chars]))

    return turns

In [4]:
def generate_rejected_batch(layer, prompts, lens, pad_token=0, temperature=0.8, top_k=20):
    sequences = [list(p) for p in prompts]

    with Tensor.no_grad():
        for _ in range(max(lens)):
            windows = [(([pad_token] * CONTEXT_SIZE) + seq)[-CONTEXT_SIZE:] for seq in sequences]
            batch = np.array(windows, dtype=np.int64)
            logits = layer(Tensor(batch))
            next_logits = logits.data[:, -1, :] / temperature

            k = min(top_k, next_logits.shape[-1])
            for row in range(len(prompts)):
                row_logits = next_logits[row]
                threshold = np.partition(row_logits, -k)[-k]
                row_logits = np.where(row_logits < threshold, -np.inf, row_logits)
                probs = np.exp(row_logits - np.max(row_logits))
                probs /= np.sum(probs)
                sequences[row].append(int(np.random.choice(len(probs), p=probs)))

    return [seq[len(prompts[i]): len(prompts[i]) + lens[i]] for i, seq in enumerate(sequences)]

In [5]:
DATA_FILE = "../../../tinyshakespeare.txt"
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"
DPO_SAMPLES = "../../dpo-samples.pkl"

In [6]:
LEARNING_RATE = 0.001
BATCH_SIZE = 4
CONTEXT_SIZE = 32
EMBEDDING_SIZE = 64
HEADS = 2
BLOCKS = 2

In [7]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)

layer = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
model = GPTModel(layer, None, None)
model.load(MODEL_FILE)
layer.eval()

pairs = list(pairwise(extract_turns()))
np.random.shuffle(pairs)
pairs = pairs[:1024]

samples = []
for start in range(0, len(pairs), 64):
    chunk = pairs[start: start + 64]

    prompt_tokens = [dataset.encode(f"{a}:\n{t}\n") for (a, t), (_, _) in chunk]
    chosen_tokens = [dataset.encode(f"{b}:\n{t}\n") for (_, _), (b, t) in chunk]

    lens = [len(c) for c in chosen_tokens]
    rejected_tokens = generate_rejected_batch(layer, prompt_tokens, lens)

    for p, c, r in zip(prompt_tokens, chosen_tokens, rejected_tokens):
        if len(r) == 0 or r == c:
            continue
        samples.append((p, c, r))

    print(f"generated {len(samples)}/{len(pairs)} DPO samples")

with open(DPO_SAMPLES, "wb") as f:
    pickle.dump(samples, f)
print(f"saved {len(samples)} DPO samples to {DPO_SAMPLES}")

generated 64/1024 DPO samples
generated 128/1024 DPO samples
generated 192/1024 DPO samples
generated 256/1024 DPO samples
generated 320/1024 DPO samples
generated 384/1024 DPO samples
generated 448/1024 DPO samples
generated 512/1024 DPO samples
generated 576/1024 DPO samples
generated 640/1024 DPO samples
generated 704/1024 DPO samples
generated 768/1024 DPO samples
generated 832/1024 DPO samples
generated 896/1024 DPO samples
generated 960/1024 DPO samples
generated 1024/1024 DPO samples
saved 1024 DPO samples to ../../dpo-samples.pkl


In [8]:
sample = samples[0]
print(f"{dataset.decode(sample[0])}\n{dataset.decode(sample[1])}\n{dataset.decode(sample[2])}")

MENENIUS:
I neither care for the world nor your general: for such things as you, I can scarce think there's an

First Senator:
A noble fellow, I warrant him.

that but to it ladle noplece.

DUKE VINCENTIO:
